GSE124395: Human Liver Cell Atlas

source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE124395

In [ ]:
!pip install aeacus

In [ ]:
import os
import tarfile
import gzip
import scanpy as sc
import anndata as ad
import aeacus
import numpy as np
import pandas as pd

In [12]:
DATA_DIR = '/home/pandeyps/Prefix/aeacus/data/GSE124395'
os.makedirs(DATA_DIR, exist_ok=True)
os.chdir(DATA_DIR)

In [ ]:
RAW_URL = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE124nnn/GSE124395/suppl/GSE124395_RAW.tar'
DATA_URL = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE124nnn/GSE124395/suppl/GSE124395_Normalhumanlivercellatlasdata.txt.gz'
CLUSTER_URL = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE124nnn/GSE124395/suppl/GSE124395_clusterpartition.txt.gz'

RAW_TAR = os.path.join(DATA_DIR, 'GSE124395_RAW.tar')
DATA_FILE = os.path.join(DATA_DIR, 'GSE124395_Normalhumanlivercellatlasdata.txt.gz')
CLUSTER_FILE = os.path.join(DATA_DIR, 'GSE124395_clusterpartition.txt.gz')

!curl -L -o {RAW_TAR} {RAW_URL}
!curl -L -o {DATA_FILE} {DATA_URL}
!curl -L -o {CLUSTER_FILE} {CLUSTER_URL}

In [ ]:
with tarfile.open(RAW_TAR, 'r') as tar:
    tar.extractall(path=DATA_DIR)

In [ ]:
with gzip.open(DATA_FILE, 'rt') as f:
    for i in range(5):
        line = f.readline()

In [ ]:
with gzip.open(DATA_FILE, 'rt') as f:
    header = f.readline()
    print(f"Header: {header[:200]}...")

with gzip.open(DATA_FILE, 'rt') as f:
    df = pd.read_csv(f, sep='\t', index_col=0)

In [ ]:
if df.shape[0] > df.shape[1]:
    print(f" genes ({df.shape[0]}) x cells ({df.shape[1]}); ") #needs transpose here.
    df_t = df.T
else:
    print(f" cells ({df.shape[0]}) x genes ({df.shape[1]})")
    df_t = df

In [ ]:
adata = ad.AnnData(X=df_t.values, obs=pd.DataFrame(index=df_t.index), var=pd.DataFrame(index=df_t.columns))

In [ ]:
if not adata.obs_names.is_unique:
    adata.obs_names_make_unique()
if not adata.var_names.is_unique:
    adata.var_names_make_unique()

In [ ]:
with gzip.open(CLUSTER_FILE, 'rt') as f:
    cluster_df = pd.read_csv(f, sep='\t', index_col=0)

# merge
common = adata.obs.index.intersection(cluster_df.index)
if len(common) > 0:
    for col in cluster_df.columns:
        adata.obs[col] = cluster_df.loc[common, col].reindex(adata.obs.index)

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
TEMP_H5AD = os.path.join(DATA_DIR, 'GSE124395_combined.h5ad')
adata.write_h5ad(TEMP_H5AD)

In [22]:
result = aeacus.Profiler(test_input=TEMP_H5AD, norm_type="cpm_log1p").load().profile()
print(f"Malignant cells: {(result.obs['malignancy_call'] == 'Malignant').sum()}")
print(f"Non-malignant cells: {(result.obs['malignancy_call'] == 'Normal').sum()}")

Model features: 3778
Missing features: 45 (1.19%)
Malignant cells: 711
Non-malignant cells: 11350
